## ETL Project

### 1. EXTRACT (Ingestion)

- I used the read_files function because it is the most modern and reliable way to load data in Databricks. 
- Since the datasets live in Unity Catalog Volumes, it instantly recognises the data stored there and automatically identify the correct column data types (like dates and numbers) directly from the CSV files. 
- Using DROP TABLE IF EXISTS supports idempotency where anyone can run your notebook from top to bottom multiple times without it breaking.

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS bronze_hospital;
USE bronze_hospital;

In [0]:
%sql

-- Patients Ingestion
DROP TABLE IF EXISTS patients;
CREATE TABLE patients AS
SELECT * FROM read_files(
  "/Volumes/workspace/default/hospital_dataset/patients.csv",
  format => "csv",
  header => true,
  inferSchema => true);

-- Encounters Ingestion
DROP TABLE IF EXISTS encounters;
CREATE TABLE encounters AS
SELECT * FROM read_files(
  "/Volumes/workspace/default/hospital_dataset/encounters.csv",
  format => "csv",
  header => true,
  inferSchema => true);

-- Diagnoses Ingestion
DROP TABLE IF EXISTS diagnoses;
CREATE TABLE diagnoses AS 
SELECT * FROM read_files(
  "/Volumes/workspace/default/hospital_dataset/diagnoses.csv", 
  format => "csv", 
  header => true, 
  inferSchema => true);

-- Procedures Ingestion
DROP TABLE IF EXISTS procedures;
CREATE TABLE procedures AS 
SELECT * FROM read_files(
  "/Volumes/workspace/default/hospital_dataset/procedures.csv", 
  format => "csv", 
  header => true, 
  inferSchema => true);

-- Medications Ingestion
DROP TABLE IF EXISTS medications;
CREATE TABLE medications AS 
SELECT * FROM read_files(
  "/Volumes/workspace/default/hospital_dataset/medications.csv", 
  format => "csv", 
  header => true, 
  inferSchema => true);

-- Laboratory Tests Ingestion
DROP TABLE IF EXISTS lab_tests;
CREATE TABLE lab_tests AS 
SELECT * FROM read_files(
  "/Volumes/workspace/default/hospital_dataset/lab_tests.csv", 
  format => "csv", 
  header => true, 
  inferSchema => true);

-- Claims and Billings Ingestion
DROP TABLE IF EXISTS claims_and_billing;
CREATE TABLE claims_and_billing AS 
SELECT * FROM read_files(
  "/Volumes/workspace/default/hospital_dataset/claims_and_billing.csv", 
  format => "csv", 
  header => true, 
  inferSchema => true);

-- Providers Ingestion
DROP TABLE IF EXISTS providers;
CREATE TABLE providers AS 
SELECT * FROM read_files(
  "/Volumes/workspace/default/hospital_dataset/providers.csv", 
  format => "csv", 
  header => true, 
  inferSchema => true);

-- Denials Ingestion
DROP TABLE IF EXISTS denials;
CREATE TABLE denials AS 
SELECT * FROM read_files(
  "/Volumes/workspace/default/hospital_dataset/denials.csv", 
  format => "csv", 
  header => true, 
  inferSchema => true);


In [0]:
%sql
-- Verify tables
SHOW TABLES IN bronze_hospital;

### 2. TRANSFORM

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS silver_hospital;


2.1. Cleaning Patients Table

In [0]:
%sql

-- Check data quality
SELECT * FROM bronze_hospital.patients
LIMIT 100;


Observations and considerations:
1. patient_id, a key_identifier starts with PAT and six numbers
2. Ensure first and last name formats are consistent (First, Last)
3. DOB and dates to be consistent as dates
4. For categorical entries (eg. Gender, Ethnicity, etc), to ensure categories reflect the real data gap, we will treat 'Unknown' catgeory as NULL.
5. Ensure emails are consistent in format (lowercase)
6. Consider calculating age using current date and DOB for later analysis
7. Remove rows with null patient_id
8. Phone numbers are not consistent, some have +1, others (), then some with . in between numbers, will remove non-numeric char


COMPLETENESS CHECK

I checked for NULLs. Using CTE to flag data issues and calculate percentages.

In [0]:
%sql

-- '1' if the data is GOOD, and '0' if it is BAD (NULL or 'Unknown')
WITH patients_quality_flags AS (
  SELECT
    -- Ids
    CASE WHEN patient_id IS NULL THEN 1 ELSE 0 END AS null_id,
  
    -- Demographics
    CASE WHEN first_name IS NULL THEN 1 ELSE 0 END AS null_first_name,
    CASE WHEN last_name IS NULL THEN 1 ELSE 0 END AS null_last_name,
    CASE WHEN dob IS NULL THEN 1 ELSE 0 END AS null_dob,
    CASE WHEN gender IS NULL OR gender = 'Unknown' THEN 1 ELSE 0 END AS null_gender,
    CASE WHEN ethnicity IS NULL OR ethnicity = 'Unknown' THEN 1 ELSE 0 END AS null_ethnicity,
  
    -- Contact info
    CASE WHEN phone IS NULL THEN 1 ELSE 0 END AS null_phone,
    CASE WHEN email IS NULL THEN 1 ELSE 0 END AS null_email,
  
    -- Location
    CASE WHEN address IS NULL THEN 1 ELSE 0 END AS null_address,
    CASE WHEN city IS NULL THEN 1 ELSE 0 END AS null_city,
    CASE WHEN state IS NULL THEN 1 ELSE 0 END AS null_state,
    CASE WHEN zip IS NULL THEN 1 ELSE 0 END AS null_zip,
  
    -- Social/Insurance
    CASE WHEN marital_status IS NULL OR marital_status = 'Unknown' THEN 1 ELSE 0 END AS null_marital,
    CASE WHEN insurance_type IS NULL OR insurance_type = 'Unknown' THEN 1 ELSE 0 END AS null_insurance,
    CASE WHEN registration_date IS NULL THEN 1 ELSE 0 END AS null_registration_date

FROM bronze_hospital.patients
)

-- Calculate percentages of nulls in each column
SELECT
  COUNT(*) AS total_rows,
  ROUND(AVG(null_id) * 100, 2) AS pct_null_id,
  ROUND(AVG(null_first_name) * 100, 2) AS pct_null_first_name,
  ROUND(AVG(null_last_name) * 100, 2) AS pct_null_last_name,
  ROUND(AVG(null_dob) * 100, 2) AS pct_null_dob,
  ROUND(AVG(null_gender) * 100, 2) AS pct_null_gender,
  ROUND(AVG(null_ethnicity) * 100, 2) AS pct_null_ethnicity,
  ROUND(AVG(null_phone) * 100, 2) AS pct_null_phone,
  ROUND(AVG(null_email) * 100, 2) AS pct_null_email,
  ROUND(AVG(null_address) * 100, 2) AS pct_null_address,
  ROUND(AVG(null_city) * 100, 2) AS pct_null_city,
  ROUND(AVG(null_state) * 100, 2) AS pct_null_state,
  ROUND(AVG(null_zip) * 100, 2) AS pct_null_zip,
  ROUND(AVG(null_marital) * 100, 2) AS pct_null_marital,
  ROUND(AVG(null_insurance) * 100, 2) AS pct_null_insurance,
  ROUND(AVG(null_registration_date) * 100, 2) AS pct_null_registration_date
FROM patients_quality_flags;
    


- There were 60,000 unique instances/rows.
- Phone, Email, Address, City, State, Zip, Marital Status are columns with NULLS.
- Only Email has NULLs accounting to ~ 20%, the rest have NULLs less than 10%.

UNIQUENESS VALIDATION

I check for duplicates in patient_id as this is crucial in analysis.

In [0]:
%sql

-- Check duplicates
SELECT COUNT(*)
FROM bronze_hospital.patients
GROUP BY patient_id
HAVING COUNT(*) > 1;


There are no patient duplicates.

PERFORMED DATA CLEANING
- Normalise strings and dates
- Ensure correct data types
- Create age column for analysis use

*This will be applied to all tables extracted/ingested.


In [0]:
%sql

-- Cleaning Patients Table
CREATE OR REPLACE TABLE silver_hospital.patients AS
SELECT
  TRIM(patient_id) as patient_id,
  INITCAP(TRIM(first_name)) AS first_name,
  INITCAP(TRIM(last_name)) AS last_name,

  -- TRY_CAST for safer date conversions
  TRY_CAST(dob AS DATE) AS birth_date,
  
  -- Age calculation: using TRY_CAST ensures 'Unknown' dates don't break the math
  FLOOR(DATEDIFF(CURRENT_DATE(), TRY_CAST(dob AS DATE)) / 365.25) AS age,
  
  NULLIF(gender, 'Unknown') AS gender,
  NULLIF(ethnicity, 'Unknown') AS ethnicity,
  NULLIF(insurance_type, 'Unknown') AS insurance_type,
  NULLIF(marital_status, 'Unknown') AS marital_status,
  NULLIF(address, 'Unknown') AS address,
  NULLIF(city, 'Unknown') AS city,
  NULLIF(state, 'Unknown') AS state,
  
  -- zip encountered BIGINT errors; handle as STRING first
  NULLIF(TRIM(CAST(zip AS STRING)), 'Unknown') AS zip, 
  
  -- Normalise phone numbers and ensure they remain strings
  regexp_replace(CAST(phone AS STRING), '[^0-9]', '') AS phone, 
  
  NULLIF(email, 'Unknown') AS email,
  TRY_CAST(registration_date AS DATE) as registration_date

FROM bronze_hospital.patients

-- Filter out records with no ID
WHERE patient_id IS NOT NULL 
  AND patient_id != 'Unknown';

-- Check data quality
SELECT * FROM silver_hospital.patients
LIMIT 100;



2.2. Cleaning Encounters Table

In [0]:
%sql

-- Check table
SELECT *
FROM bronze_hospital.encounters
LIMIT 100;


Observations:
- encounter_id, patient_id and provider_id has consistent format (3 LETTERS followed by numbers)
- there are dates in this table, we will ensure it is in date format
- there are no unusual formatting errors that stood out
- lots of NULLs with admission type, discharge date, length of stay
- numeric columns are in their correct data type
- visit type can be: Inpatients, Outpatient, Emergency. Only Inpatients have completed fields for admission, discharge, and length of stay.
- diagnosis code are ICD-10 diagnosis codes 
- department, reason for visit and diagnosis code have normalised formatting

In [0]:
%sql

WITH encounters_quality_flags AS (
SELECT
  CASE WHEN encounter_id IS NULL THEN 1 ELSE 0 END AS null_encounter_id,
  CASE WHEN patient_id IS NULL THEN 1 ELSE 0 END AS null_patient_id,
  CASE WHEN provider_id IS NULL THEN 1 ELSE 0 END AS null_provider_id,
  CASE WHEN visit_date IS NULL THEN 1 ELSE 0 END AS null_visit_date,
  CASE WHEN visit_type IS NULL THEN 1 ELSE 0 END AS null_visit_type,
  CASE WHEN department IS NULL THEN 1 ELSE 0 END AS null_department,
  CASE WHEN reason_for_visit IS NULL THEN 1 ELSE 0 END AS null_reason_for_visit,
  CASE WHEN diagnosis_code IS NULL THEN 1 ELSE 0 END AS null_diagnosis_code,
  CASE WHEN admission_type IS NULL THEN 1 ELSE 0 END AS null_admission_type,
  CASE WHEN discharge_date IS NULL THEN 1 ELSE 0 END AS null_discharge_date,
  CASE WHEN length_of_stay IS NULL THEN 1 ELSE 0 END AS null_length_of_stay,
  CASE WHEN status IS NULL THEN 1 ELSE 0 END AS null_status,
  CASE WHEN readmitted_flag IS NULL THEN 1 ELSE 0 END AS null_readmitted_flag
FROM bronze_hospital.encounters
)

SELECT
  COUNT(*) AS total_rows,
  ROUND(AVG(null_encounter_id) * 100, 2) AS pct_null_encounter_id,
  ROUND(AVG(null_patient_id) * 100, 2) AS pct_null_patient_id,
  ROUND(AVG(null_provider_id) * 100, 2) AS pct_null_provider_id,
  ROUND(AVG(null_visit_date) * 100, 2) AS pct_null_visit_date,
  ROUND(AVG(null_visit_type) * 100, 2) AS pct_null_visit_type,
  ROUND(AVG(null_department) * 100, 2) AS pct_null_department,
  ROUND(AVG(null_reason_for_visit) * 100, 2) AS pct_null_reason_for_visit,
  ROUND(AVG(null_diagnosis_code) * 100, 2) AS pct_null_diagnosis_code,
  ROUND(AVG(null_admission_type) * 100, 2) AS pct_null_admission_type,
  ROUND(AVG(null_discharge_date) * 100, 2) AS pct_null_discharge_date,
  ROUND(AVG(null_length_of_stay) * 100, 2) AS pct_null_length_of_stay,
  ROUND(AVG(null_status) * 100, 2) AS pct_null_status,
  ROUND(AVG(null_readmitted_flag) * 100, 2) AS pct_null_readmitted_flag
FROM encounters_quality_flags;


In [0]:
%sql

-- Check for duplicates
SELECT COUNT(*) AS duplicated_rows
FROM bronze_hospital.encounters
GROUP BY patient_id, encounter_id
HAVING COUNT(*) > 1;


- There were 70,000 rows
- Admission type, discharge date, length of stay columns have more than 60% missing data. This may due to only Inpatients visit type have these fields completed.

In [0]:
%sql

-- Cleaning Encounters Table

CREATE OR REPLACE TABLE silver_hospital.encounters AS
SELECT
  encounter_id,
  patient_id,
  provider_id,
  TRY_CAST(visit_date AS DATE) AS visit_date,
  visit_type,
  department,
  reason_for_visit,
  diagnosis_code,
  admission_type,
  TRY_CAST(discharge_date AS DATE) as discharge_date,
  TRY_CAST(length_of_stay AS INT) as length_of_stay,
  status,
  readmitted_flag
FROM bronze_hospital.encounters
WHERE patient_id IS NOT NULL
  AND encounter_id IS NOT NULL
  AND provider_id IS NOT NULL;

-- Check data generated
SELECT * FROM silver_hospital.encounters
LIMIT 100;


I used TRY_CAST here to identify malformed string it can't convert and simply returns it as NULL.

2.3 Cleaning Diagnoses Table

In [0]:
%sql

SELECT * FROM bronze_hospital.diagnoses
LIMIT 100;


- encounter_id is the unique key here
- diagnosis table provides description to the diagnosis and code, also identifies if diagnosis is chronic or not

In [0]:
%sql

WITH diagnosis_quality_flags AS (
  SELECT 
    CASE WHEN diagnosis_id IS NULL THEN 1 ELSE 0 END AS null_diagnosis_id,
    CASE WHEN encounter_id IS NULL THEN 1 ELSE 0 END AS null_encounter_id,
    CASE WHEN diagnosis_code IS NULL THEN 1 ELSE 0 END AS null_diagnosis_code,
    CASE WHEN diagnosis_description IS NULL THEN 1 ELSE 0 END AS null_diagnosis_description,
    CASE WHEN primary_flag IS NULL THEN 1 ELSE 0 END AS null_primary_flag,
    CASE WHEN chronic_flag IS NULL THEN 1 ELSE 0 END AS null_chronic_flag
  FROM bronze_hospital.diagnoses
)

SELECT
  COUNT(*) AS total_rows,
  ROUND(AVG(null_diagnosis_id) * 100, 2) AS pct_null_diagnosis_id,
  ROUND(AVG(null_encounter_id) * 100, 2) AS pct_null_encounter_id,
  ROUND(AVG(null_diagnosis_code) * 100, 2) AS pct_null_diagnosis_code,
  ROUND(AVG(null_diagnosis_description) * 100, 2) AS pct_null_diagnosis_description,
  ROUND(AVG(null_primary_flag) * 100, 2) AS pct_null_primary_flag,
  ROUND(AVG(null_chronic_flag) * 100, 2) AS pct_null_chronic_flag
FROM diagnosis_quality_flags;



In [0]:
%sql

-- Use encounter_id to check for duplicates
SELECT COUNT(*) AS duplicated_rows
FROM bronze_hospital.diagnoses
GROUP BY encounter_id
HAVING COUNT(*) > 1


- There are no duplicates and missing data on diagnosis table.

In [0]:
%sql

CREATE OR REPLACE TABLE silver_hospital.diagnoses AS
SELECT
  diagnosis_id,
  encounter_id,
  diagnosis_code,
  diagnosis_description,
  INITCAP(TRIM(primary_flag)) AS primary_flag,  -- Standardise entry to capitalised first letter
  INITCAP(TRIM(chronic_flag)) AS chronic_flag
FROM bronze_hospital.diagnoses
WHERE diagnosis_id IS NOT NULL
  AND encounter_id IS NOT NULL;


2.4 Cleaning Procedures Table

In [0]:
%sql

SELECT * FROM bronze_hospital.procedures
LIMIT 100;

In [0]:
%sql

WITH procedure_quality_flags AS (
  SELECT 
    CASE WHEN procedure_id IS NULL THEN 1 ELSE 0 END AS null_procedure_id,
    CASE WHEN encounter_id IS NULL THEN 1 ELSE 0 END AS null_encounter_id,
    CASE WHEN procedure_code IS NULL THEN 1 ELSE 0 END AS null_procedure_code,
    CASE WHEN procedure_description IS NULL THEN 1 ELSE 0 END AS null_procedure_description,
    CASE WHEN procedure_date IS NULL THEN 1 ELSE 0 END AS null_procedure_date,
    CASE WHEN provider_id IS NULL THEN 1 ELSE 0 END AS null_provider_id,
    CASE WHEN procedure_cost IS NULL THEN 1 ELSE 0 END AS null_procedure_cost
  FROM bronze_hospital.procedures
)

SELECT
  COUNT(*) AS total_rows,
  ROUND(AVG(null_procedure_id) * 100, 2) AS pct_null_procedure_id,
  ROUND(AVG(null_encounter_id) * 100, 2) AS pct_null_encounter_id,
  ROUND(AVG(null_procedure_code) * 100, 2) AS pct_null_procedure_code,
  ROUND(AVG(null_procedure_description) * 100, 2) AS pct_null_procedure_description,
  ROUND(AVG(null_procedure_date) * 100, 2) AS pct_null_procedure_date,
  ROUND(AVG(null_provider_id) * 100, 2) AS pct_null_provider_id,
  ROUND(AVG(null_procedure_cost) * 100, 2) AS pct_null_procedure_cost
FROM procedure_quality_flags;


In [0]:
%sql
-- Check duplicates using procedure_code and encounter_id
SELECT COUNT(*) AS duplicated_rows
FROM bronze_hospital.procedures
GROUP BY encounter_id, procedure_id
HAVING COUNT(*) > 1;


In [0]:
%sql

CREATE OR REPLACE TABLE silver_hospital.procedures
SELECT
  procedure_id,
  encounter_id,
  procedure_code,
  procedure_description,
  TRY_CAST(procedure_date AS DATE) AS procedure_date,
  provider_id,
  TRY_CAST(procedure_cost AS DOUBLE) AS procedure_cost
FROM bronze_hospital.procedures
WHERE procedure_id IS NOT NULL
  AND encounter_id IS NOT NULL;


2.5 Cleaning Medications Table

In [0]:
%sql

SELECT * FROM bronze_hospital.medications
LIMIT 100;


- dosage is formatted as string with units
- duration is formatted as string (number and days/weeks/cycles). Hard the standardise this given different frequencies and duration for different drugs

In [0]:
%sql

WITH medication_quality_flags AS (
  SELECT 
    CASE WHEN medication_id IS NULL THEN 1 ELSE 0 END AS null_medication_id,
    CASE WHEN encounter_id IS NULL THEN 1 ELSE 0 END AS null_encounter_id,
    CASE WHEN drug_name IS NULL THEN 1 ELSE 0 END AS null_drug_name,
    CASE WHEN dosage IS NULL THEN 1 ELSE 0 END AS null_dosage,
    CASE WHEN route IS NULL THEN 1 ELSE 0 END AS null_route,
    CASE WHEN frequency IS NULL THEN 1 ELSE 0 END AS null_frequency,
    CASE WHEN duration IS NULL THEN 1 ELSE 0 END AS null_duration,
    CASE WHEN prescribed_date IS NULL THEN 1 ELSE 0 END AS null_prescribed_date,
    CASE WHEN prescriber_id IS NULL THEN 1 ELSE 0 END AS null_prescriber_id,
    CASE WHEN cost IS NULL THEN 1 ELSE 0 END AS null_cost
  FROM bronze_hospital.medications
)
SELECT
  COUNT(*) AS total_rows,
  ROUND(AVG(null_medication_id) * 100, 2) AS pct_null_medication_id,
  ROUND(AVG(null_encounter_id) * 100, 2) AS pct_null_encounter_id,
  ROUND(AVG(null_drug_name) * 100, 2) AS pct_null_drug_name,
  ROUND(AVG(null_dosage) * 100, 2) AS pct_null_dosage,
  ROUND(AVG(null_route) * 100, 2) AS pct_null_route,
  ROUND(AVG(null_frequency) * 100, 2) AS pct_null_frequency,
  ROUND(AVG(null_duration) * 100, 2) AS pct_null_duration,
  ROUND(AVG(null_prescribed_date) * 100, 2) AS pct_null_prescribed_date,
  ROUND(AVG(null_prescriber_id) * 100, 2) AS pct_null_prescriber_id,
  ROUND(AVG(null_cost) * 100, 2) AS pct_null_cost
FROM medication_quality_flags;


In [0]:
%sql

SELECT COUNT(*) AS duplicated_rows
FROM bronze_hospital.medications
GROUP BY medication_id, encounter_id, drug_name
HAVING COUNT(*) > 1;


- There are no duplicates using drug name, medication_id and encounter_id as filters.
- There are approx 94,000 unique rows.

In [0]:
%sql

CREATE OR REPLACE TABLE silver_hospital.medications
SELECT
  medication_id,
  encounter_id,
  drug_name,
  dosage,
  route,
  frequency,
  duration,
  TRY_CAST(prescribed_date AS DATE) AS prescribed_date,
  prescriber_id,
  TRY_CAST(cost AS DOUBLE) AS cost
FROM bronze_hospital.medications
WHERE medication_id IS NOT NULL
  AND encounter_id IS NOT NULL;


2.6 Cleaning Laboratory Table

In [0]:
%sql

SELECT * FROM bronze_hospital.lab_tests
LIMIT 100;


- encounter_id is the unique key
- lab_id, encounter_id, test_code has a standard format
- specimen_type has entry 'Unknown', will consider NULL as it's likely missing on entry and not an Unknown option
- test_result can be Normal, Abnormal, Preliminary
- units and normal_range has lots of 'N/A', will convert to NULL
- only date format is for test_date


In [0]:
%sql

WITH lab_tests_quality_flags AS (
  SELECT 
    CASE WHEN lab_id IS NULL THEN 1 ELSE 0 END AS null_lab_id,
    CASE WHEN encounter_id IS NULL THEN 1 ELSE 0 END AS null_encounter_id,
    CASE WHEN test_name IS NULL THEN 1 ELSE 0 END AS null_test_name,
    CASE WHEN test_code IS NULL THEN 1 ELSE 0 END AS null_test_code,
    CASE WHEN specimen_type IS NULL OR specimen_type = 'Unknown' THEN 1 ELSE 0 END AS null_specimen_type,
    CASE WHEN test_result IS NULL THEN 1 ELSE 0 END AS null_test_result,
    CASE WHEN units IS NULL OR units = 'N/A' THEN 1 ELSE 0 END AS null_units,
    CASE WHEN normal_range IS NULL or normal_range = 'N/A' THEN 1 ELSE 0 END AS null_normal_range,
    CASE WHEN test_date IS NULL THEN 1 ELSE 0 END AS null_test_date,
    CASE WHEN status IS NULL THEN 1 ELSE 0 END AS null_status
  FROM bronze_hospital.lab_tests
)

SELECT 
  COUNT(*) AS total_rows,
  ROUND(AVG(null_lab_id) * 100, 2) AS pct_null_lab_id,
  ROUND(AVG(null_encounter_id) * 100, 2) AS pct_null_encounter_id,
  ROUND(AVG(null_test_name) * 100, 2) AS pct_null_test_name,
  ROUND(AVG(null_test_code) * 100, 2) AS pct_null_test_code,
  ROUND(AVG(null_specimen_type) * 100, 2) AS pct_null_specimen_type,
  ROUND(AVG(null_test_result) * 100, 2) AS pct_null_test_result,
  ROUND(AVG(null_units) * 100, 2) AS pct_null_units,
  ROUND(AVG(null_normal_range) * 100, 2) AS pct_null_normal_range,
  ROUND(AVG(null_test_date) * 100, 2) AS pct_null_test_date,
  ROUND(AVG(null_status) * 100, 2) AS pct_null_status
FROM lab_tests_quality_flags;


In [0]:
%sql

SELECT COUNT(*) AS duplicated_rows
FROM bronze_hospital.lab_tests
GROUP BY lab_id, encounter_id, test_name
HAVING COUNT(*) > 1;


- There are no duplicated rows (repeated test in one encounter)
- About 31% of entries in specimen type is missing
- About 72% and 94% are missing for units and normal_range column. Not really significant data so these columns can be excluded.

In [0]:
%sql

CREATE OR REPLACE TABLE silver_hospital.lab_tests
SELECT
  lab_id,
  encounter_id,
  test_name,
  test_code,
  NULLIF(specimen_type, 'Unknown') AS specimen_type,
  test_result,
  NULLIF(units, 'N/A') AS units,
  NULLIF(normal_range, 'N/A') AS normal_range,
  TRY_CAST(test_date AS DATE) AS test_date,
  status
FROM bronze_hospital.lab_tests
WHERE lab_id IS NOT NULL
  AND encounter_id IS NOT NULL;

SELECT * FROM silver_hospital.lab_tests
LIMIT 100;
  

2.7 Cleaning Claims and Billings Table

In [0]:
%sql
SELECT * FROM bronze_hospital.claims_and_billing
LIMIT 100;

In [0]:
%sql

WITH claims_billings_quality_flags AS (
  SELECT 
    CASE WHEN billing_id IS NULL THEN 1 ELSE 0 END AS null_billing_id,
    CASE WHEN patient_id IS NULL THEN 1 ELSE 0 END AS null_patient_id,
    CASE WHEN encounter_id IS NULL THEN 1 ELSE 0 END AS null_encounter_id,
    CASE WHEN insurance_provider IS NULL THEN 1 ELSE 0 END AS null_insurance_provider,
    CASE WHEN payment_method IS NULL THEN 1 ELSE 0 END AS null_payment_method,
    CASE WHEN claim_id IS NULL THEN 1 ELSE 0 END AS null_claim_id,
    CASE WHEN claim_billing_date IS NULL THEN 1 ELSE 0 END AS null_claim_billing_date,
    CASE WHEN billed_amount IS NULL THEN 1 ELSE 0 END AS null_billed_amount,
    CASE WHEN paid_amount IS NULL THEN 1 ELSE 0 END AS null_paid_amount,
    CASE WHEN claim_status IS NULL THEN 1 ELSE 0 END AS null_claim_status,
    CASE WHEN denial_reason IS NULL THEN 1 ELSE 0 END AS null_denial_reason
  FROM bronze_hospital.claims_and_billing
)
SELECT 
  COUNT(*) AS total_rows,
  ROUND(AVG(null_billing_id) * 100, 2) AS pct_null_billing_id,
  ROUND(AVG(null_patient_id) * 100, 2) AS pct_null_patient_id,
  ROUND(AVG(null_encounter_id) * 100, 2) AS pct_null_encounter_id,
  ROUND(AVG(null_insurance_provider) * 100, 2) AS pct_null_insurance_provider,
  ROUND(AVG(null_payment_method) * 100, 2) AS pct_null_payment_method,
  ROUND(AVG(null_claim_id) * 100, 2) AS pct_null_claim_id,
  ROUND(AVG(null_claim_billing_date) * 100, 2) AS pct_null_claim_billing_date,    
  ROUND(AVG(null_billed_amount) * 100, 2) AS pct_null_billed_amount,
  ROUND(AVG(null_paid_amount) * 100, 2) AS pct_null_paid_amount,
  ROUND(AVG(null_claim_status) * 100, 2) AS pct_null_claim_status,
  ROUND(AVG(null_denial_reason) * 100, 2) AS pct_null_denial_reason
FROM claims_billings_quality_flags;


- approx 15% null on Claim ID and claim billing date
- only 10% have completed reason for denial (only denial cases)
- claim_billing_date is on datetime format, will only extract date

In [0]:
%sql
SELECT COUNT(*) AS duplicated_rows
FROM bronze_hospital.claims_and_billing
GROUP BY billing_id
HAVING COUNT(*) > 1;


In [0]:
%sql
-- Check format of claim_billing_date as returning nulls
SELECT 
  claim_billing_date,
  TYPEOF(claim_billing_date) AS data_type,
  LENGTH(claim_billing_date) AS length
FROM bronze_hospital.claims_and_billing
LIMIT 10;

- claim_billing_date is a string with 16 characters, which is why the date conversion was failing

In [0]:
%sql
CREATE OR REPLACE TABLE silver_hospital.claims_and_billing AS
SELECT
  billing_id,
  patient_id,
  encounter_id,
  insurance_provider,
  payment_method,
  claim_id,
  -- Extract first 10 characters and convert string to date format
  TO_DATE(LEFT(claim_billing_date, 10), 'dd-MM-yyyy') AS claim_billing_date,
  billed_amount,
  paid_amount,
  ROUND((billed_amount - paid_amount), 2) AS balance,
  claim_status,
  denial_reason
FROM bronze_hospital.claims_and_billing
WHERE billing_id IS NOT NULL;

In [0]:
%sql
SELECT * FROM silver_hospital.claims_and_billing
LIMIT 10;

2.8 Cleaning Providers Table

In [0]:
%sql
SELECT * FROM bronze_hospital.providers
LIMIT 100;


In [0]:
%sql
WITH providers_flags AS (
  SELECT
    CASE WHEN provider_id IS NULL THEN 1 ELSE 0 END AS null_provider_id,
    CASE WHEN name IS NULL THEN 1 ELSE 0 END AS null_name,
    CASE WHEN department IS NULL THEN 1 ELSE 0 END AS null_department,
    CASE WHEN specialty IS NULL THEN 1 ELSE 0 END AS null_specialty,
    CASE WHEN npi IS NULL THEN 1 ELSE 0 END AS null_npi,
    CASE WHEN inhouse IS NULL THEN 1 ELSE 0 END AS null_inhouse,
    CASE WHEN location IS NULL THEN 1 ELSE 0 END AS null_location,
    CASE WHEN years_experience IS NULL THEN 1 ELSE 0 END AS null_years_experience,
    CASE WHEN contact_info IS NULL THEN 1 ELSE 0 END AS null_contact_info,
    CASE WHEN email IS NULL THEN 1 ELSE 0 END AS null_email
  FROM bronze_hospital.providers
)
SELECT
  COUNT(*) AS total_rows,
  ROUND(AVG(null_provider_id) * 100, 2) AS pct_null_provider_id,
  ROUND(AVG(null_name) * 100, 2) AS pct_null_name,
  ROUND(AVG(null_department) * 100, 2) AS pct_null_department,
  ROUND(AVG(null_specialty) * 100, 2) AS pct_null_specialty,
  ROUND(AVG(null_npi) * 100, 2) AS pct_null_npi,
  ROUND(AVG(null_inhouse) * 100, 2) AS pct_null_inhouse,
  ROUND(AVG(null_location) * 100, 2) AS pct_null_location,
  ROUND(AVG(null_years_experience) * 100, 2) AS pct_null_years_experience,
  ROUND(AVG(null_contact_info) * 100, 2) AS pct_null_contact_info,
  ROUND(AVG(null_email) * 100, 2) AS pct_null_email
FROM providers_flags;


- There are 1491 rows on this dataset.
- Contact info and email have missing data (10% and 21% respectively).

In [0]:
%sql
SELECT COUNT(*) AS duplicated_rows
FROM bronze_hospital.providers
GROUP BY provider_id
HAVING COUNT(*) > 1;


In [0]:
%sql
-- Removed name column, provider_ids are unique
CREATE OR REPLACE TABLE silver_hospital.providers AS
SELECT
  provider_id,
  department,
  specialty,
  npi,
  inhouse,
  location,
  years_experience,
  contact_info,
  email
FROM bronze_hospital.providers
WHERE provider_id IS NOT NULL;


2.9 Cleaning Denials Table

In [0]:
%sql
SELECT * FROM bronze_hospital.denials
LIMIT 100;


In [0]:
%sql
WITH denials_flags AS (
  SELECT
    CASE WHEN claim_id IS NULL THEN 1 ELSE 0 END AS null_claim_id,
    CASE WHEN denial_id  IS NULL THEN 1 ELSE 0 END AS null_denial_id,
    CASE WHEN denial_reason_code IS NULL THEN 1 ELSE 0 END AS null_denial_reason_code,
    CASE WHEN denial_reason_description IS NULL THEN 1 ELSE 0 END AS null_denial_reason_descr,
    CASE WHEN denied_amount IS NULL THEN 1 ELSE 0 END AS null_denied_amount,
    CASE WHEN denial_date IS NULL THEN 1 ELSE 0 END AS null_denial_date,
    CASE WHEN appeal_filed IS NULL THEN 1 ELSE 0 END AS null_appeal_filed,
    CASE WHEN appeal_status IS NULL THEN 1 ELSE 0 END AS null_appeal_status,
    CASE WHEN appeal_resolution_date IS NULL THEN 1 ELSE 0 END AS null_appeal_resolution_date,
    CASE WHEN final_outcome IS NULL THEN 1 ELSE 0 END AS null_final_outcome
  FROM bronze_hospital.denials
)

SELECT
  COUNT(*) AS total_rows,
  ROUND(AVG(null_claim_id) * 100, 2) AS pct_null_claim_id,
  ROUND(AVG(null_denial_id) * 100, 2) AS pct_null_denial_id,
  ROUND(AVG(null_denial_reason_code) * 100, 2) AS pct_null_denial_reason_code,
  ROUND(AVG(null_denial_reason_descr) * 100, 2) AS pct_null_denial_reason_descr,
  ROUND(AVG(null_denied_amount) * 100, 2) AS pct_null_denied_amount,  
  ROUND(AVG(null_denial_date) * 100, 2) AS pct_null_denial_date,
  ROUND(AVG(null_appeal_filed) * 100, 2) AS pct_null_appeal_filed,
  ROUND(AVG(null_appeal_status) * 100, 2) AS pct_null_appeal_status,
  ROUND(AVG(null_appeal_resolution_date) * 100, 2) AS pct_null_appeal_resolution_date,
  ROUND(AVG(null_final_outcome) * 100, 2) AS pct_null_final_outcome
FROM denials_flags; 


- There are 5998 rows recording claims that are denied
- Appeal status, appeal resolution date and final outcome has 10% missing data
- Dates and integers are casted to ensure consist data types

In [0]:
%sql
SELECT COUNT(*) AS duplicated_rows
FROM bronze_hospital.denials
GROUP BY claim_id, denial_id
HAVING COUNT(*) > 1;


- No duplicate records

In [0]:
%sql
CREATE OR REPLACE TABLE silver_hospital.denials AS
SELECT
  claim_id,
  denial_id,
  denial_reason_code,
  denial_reason_description,
  TRY_CAST(denied_amount AS DOUBLE) AS denied_amount,
  TRY_CAST(denial_date AS DATE) AS denial_date,
  appeal_filed,
  appeal_status,
  TRY_CAST(appeal_resolution_date AS DATE) AS appeal_resolution_date,
  final_outcome
FROM bronze_hospital.denials;


### 3. LOAD (For Business Analytics)
The gold layer contains denormalised, business-ready tables optimised for analytics, reporting, and dashboards. These tables combine data from multiple silver tables and include calculated metrics.

In [0]:
%sql
-- Create schema for gold layer
CREATE SCHEMA IF NOT EXISTS gold_hospital;

3.1 Patient Profiles

Creates a comprehensive patient profile combining demographics, encounter history, and financial metrics.

In [0]:
%sql
CREATE OR REPLACE TABLE gold_hospital.patient_profile AS
WITH patient_encounters AS (
  SELECT 
    patient_id,
    COUNT(DISTINCT encounter_id) AS total_encounters,
    MAX(visit_date) AS last_visit_date,
    MIN(visit_date) AS first_visit_date,
    SUM(CASE WHEN readmitted_flag = 'Yes' THEN 1 ELSE 0 END) AS readmission_count,
    AVG(length_of_stay) AS avg_length_of_stay
  FROM silver_hospital.encounters
  GROUP BY patient_id
),
patient_financials AS (
  SELECT
    p.patient_id,
    SUM(cb.billed_amount) AS total_billed,
    SUM(cb.paid_amount) AS total_paid,
    SUM(cb.balance) AS total_balance,
    COUNT(DISTINCT cb.claim_id) AS total_claims
  FROM silver_hospital.patients p
  LEFT JOIN silver_hospital.encounters e ON p.patient_id = e.patient_id
  LEFT JOIN silver_hospital.claims_and_billing cb ON e.encounter_id = cb.encounter_id
  GROUP BY p.patient_id
)
SELECT
-- Removed patient name
  p.patient_id,
  p.age,
  p.gender,
  p.ethnicity,
  p.marital_status,
  p.city,
  p.state,
  p.insurance_type,
  pe.total_encounters,
  pe.first_visit_date,
  pe.last_visit_date,
  pe.readmission_count,
  ROUND(pe.avg_length_of_stay, 2) AS avg_length_of_stay,
  COALESCE(pf.total_billed, 0) AS total_billed,
  COALESCE(pf.total_paid, 0) AS total_paid,
  COALESCE(pf.total_balance, 0) AS total_outstanding_balance,
  COALESCE(pf.total_claims, 0) AS total_claims
FROM silver_hospital.patients p
LEFT JOIN patient_encounters pe ON p.patient_id = pe.patient_id
LEFT JOIN patient_financials pf ON p.patient_id = pf.patient_id;

SELECT * FROM gold_hospital.patient_profile LIMIT 10;

3.2 Visit types

a. Show many patients have presented in Emergency Dept, Outpatient, Inpatients, Telehealth

b. What are the most common reason for presentation per visit type?

In [0]:
%sql
CREATE OR REPLACE TABLE gold_hospital.presentations_type AS
-- Get counts of visit types for each month
WITH presentation_counts AS (
  SELECT
    EXTRACT(YEAR FROM visit_date) AS Year,
    EXTRACT(MONTH FROM visit_date) AS Month,
    visit_type,
    COUNT(*) AS count
  FROM silver_hospital.encounters
  GROUP BY Year, Month, visit_type
)
-- Pivot to get counts for each visit type
SELECT
  Year,
  DATE_FORMAT(MAKE_DATE(Year, Month, 1), 'MMM') AS Month_name,
  SUM(CASE WHEN visit_type = 'Emergency' THEN count ELSE 0 END) AS Emergency,
  SUM(CASE WHEN visit_type = 'Inpatients' THEN count ELSE 0 END) AS Inpatient,
  SUM(CASE WHEN visit_type = 'Outpatient' THEN count ELSE 0 END) AS Outpatient,
  SUM(CASE WHEN visit_type = 'Telehealth' THEN count ELSE 0 END) AS Telehealth,
  SUM(CASE WHEN visit_type IS NULL THEN count ELSE 0 END) AS Other,
  SUM(count) AS Total_visits
FROM presentation_counts
GROUP BY year, month
ORDER BY year, month;

    

In [0]:
%sql
SELECT *
FROM gold_hospital.presentations_type;

In [0]:
%sql
CREATE OR REPLACE TABLE gold_hospital.presentations_reason AS
-- Get top 3 reasons for visit for each month and visit type
WITH presentation_reason AS (
  SELECT
    EXTRACT(YEAR FROM visit_date) AS Year,
    EXTRACT(MONTH FROM visit_date) AS Month,
    visit_type,
    reason_for_visit,
    COUNT(*) AS count,
    -- Calculate total visits for each year/month/visit_type
    SUM(COUNT(*)) OVER (
      PARTITION BY EXTRACT(YEAR FROM visit_date), EXTRACT(MONTH FROM visit_date), visit_type
    ) AS total_visits,
    -- Rank reasons for visit for each year/month/visit_type
    RANK() OVER (
      PARTITION BY EXTRACT(YEAR FROM visit_date), EXTRACT(MONTH FROM visit_date), visit_type 
      ORDER BY COUNT(*) DESC
    ) AS rank
  FROM silver_hospital.encounters
  GROUP BY Year, Month, visit_type, reason_for_visit
)
SELECT 
  Year,
  DATE_FORMAT(MAKE_DATE(Year, Month, 1), 'MMM') AS Month_name,
  visit_type,
  reason_for_visit,
  ROUND((count * 100.00 / total_visits), 2) AS Percentage
FROM presentation_reason
WHERE rank <= 3
ORDER BY Year, Month, visit_type, rank;


In [0]:
%sql
SELECT *
FROM gold_hospital.presentations_reason;


3.3 Provider Performance

Key performance indicators for healthcare providers including patient volume, procedures, and outcomes.

In [0]:
%sql
CREATE OR REPLACE TABLE gold_hospital.provider_performance AS
WITH provider_encounters AS (
  SELECT
    provider_id,
    COUNT(DISTINCT encounter_id) AS total_encounters,
    COUNT(DISTINCT patient_id) AS unique_patients,
    AVG(length_of_stay) AS avg_los,
    SUM(CASE WHEN readmitted_flag = 'Yes' THEN 1 ELSE 0 END) AS readmissions,
    COUNT(*) AS total_visits
  FROM silver_hospital.encounters
  GROUP BY provider_id
),
provider_procedures AS (
  SELECT
    e.provider_id,
    COUNT(DISTINCT proc.procedure_id) AS total_procedures,
    SUM(proc.procedure_cost) AS total_procedure_revenue
  FROM silver_hospital.encounters e
  INNER JOIN silver_hospital.procedures proc ON e.encounter_id = proc.encounter_id
  GROUP BY e.provider_id
),
provider_financials AS (
  SELECT
    e.provider_id,
    SUM(cb.billed_amount) AS total_revenue,
    SUM(cb.paid_amount) AS total_collected
  FROM silver_hospital.encounters e
  INNER JOIN silver_hospital.claims_and_billing cb ON e.encounter_id = cb.encounter_id
  GROUP BY e.provider_id
)
SELECT
  prov.provider_id,
  prov.specialty,
  prov.department,
  prov.years_experience,
  prov.location,
  prov.inhouse,
  pe.total_encounters,
  pe.unique_patients,
  ROUND(pe.avg_los, 2) AS avg_length_of_stay,
  pe.readmissions,
  ROUND(pe.readmissions * 100.0 / NULLIF(pe.total_visits, 0), 2) AS readmission_rate_pct,
  COALESCE(pp.total_procedures, 0) AS total_procedures,
  ROUND(COALESCE(pp.total_procedure_revenue, 0), 2) AS procedure_revenue,
  ROUND(COALESCE(pf.total_revenue, 0), 2) AS total_billed,
  ROUND(COALESCE(pf.total_collected, 0), 2) AS total_collected,
  ROUND(COALESCE(pf.total_collected, 0) * 100.0 / NULLIF(pf.total_revenue, 1), 2) AS collection_rate_pct
FROM silver_hospital.providers prov
LEFT JOIN provider_encounters pe ON prov.provider_id = pe.provider_id
LEFT JOIN provider_procedures pp ON prov.provider_id = pp.provider_id
LEFT JOIN provider_financials pf ON prov.provider_id = pf.provider_id;

SELECT * FROM gold_hospital.provider_performance LIMIT 50;

3.4 Financial Analytics

Billing and claims analytics including denial rates and collection metrics.

In [0]:
%sql
CREATE OR REPLACE TABLE gold_hospital.financial_summary AS
WITH monthly_billing AS (
  SELECT
    EXTRACT(YEAR FROM claim_billing_date) AS billing_year,
    EXTRACT(MONTH FROM claim_billing_date) AS billing_month,
    insurance_provider,
    payment_method,
    COUNT(billing_id) AS total_bills,
    COUNT(claim_id) AS total_claims,
    SUM(billed_amount) AS total_billed,
    SUM(paid_amount) AS total_paid,
    SUM(balance) AS total_outstanding,
    COUNT(CASE WHEN claim_status = 'Denied' THEN 1 END) AS denied_claims,
    COUNT(CASE WHEN claim_status = 'Paid' THEN 1 END) AS approved_claims,
    COUNT(CASE WHEN claim_status IS NULL THEN 1 END) AS other
  FROM silver_hospital.claims_and_billing
  WHERE claim_billing_date IS NOT NULL
  GROUP BY billing_year, billing_month, insurance_provider, payment_method
)
SELECT
  billing_year AS Year,
  DATE_FORMAT(MAKE_DATE(billing_year, billing_month, 1), 'MMM') AS Month,
  insurance_provider,
  payment_method,
  total_bills,
  total_claims,
  ROUND(total_billed, 2) AS total_billed,
  ROUND(total_paid, 2) AS total_paid,
  ROUND(total_outstanding, 2) AS total_outstanding,
  ROUND(total_paid * 100.0 / NULLIF(total_billed, 0), 2) AS collection_rate_pct,
  denied_claims,
  approved_claims,
  other,
  ROUND(denied_claims * 100.0 / NULLIF(total_claims, 0), 2) AS denial_rate_pct,
  ROUND(approved_claims * 100.0 / NULLIF(total_claims, 0), 2) AS approval_rate_pct,
  ROUND(other * 100.0 / NULLIF(total_claims, 0), 2) AS other_rate_pct
FROM monthly_billing
ORDER BY billing_year, billing_month, total_billed DESC;


In [0]:
%sql
SELECT *
FROM gold_hospital.financial_summary;

In [0]:
%sql
-- What are the total billed amounts by provider?
SELECT
  insurance_provider,
  ROUND(SUM(billed_amount), 2) AS total_billed_amount
FROM silver_hospital.claims_and_billing
GROUP BY insurance_provider
ORDER BY total_billed_amount DESC;


In [0]:
%sql
-- How many claims were denied per payment method?
SELECT
  payment_method,
  COUNT(*) as denied_claims_count
FROM silver_hospital.claims_and_billing
WHERE claim_status = 'Denied'
GROUP BY payment_method
ORDER BY denied_claims_count DESC;


In [0]:
%sql
-- What percentage of claims are denied and approved by insurance providers?
WITH denial_counts AS (
  SELECT
    insurance_provider,
    COUNT(*) as total_claims,
    COUNT(CASE WHEN claim_status = 'Denied' THEN 1 END) as denied_claims
  FROM silver_hospital.claims_and_billing
  GROUP BY insurance_provider
), approval_counts AS (
  SELECT
    insurance_provider,
    COUNT(*) as total_claims,
    COUNT(CASE WHEN claim_status = 'Paid' THEN 1 END) as approved_claims
  FROM silver_hospital.claims_and_billing
  GROUP BY insurance_provider
)
SELECT
  denial_counts.insurance_provider,
  ROUND(denied_claims * 100.0 / NULLIF(denial_counts.total_claims, 0), 2) AS denial_rate_pct,
  ROUND(approved_claims * 100.0 / NULLIF(approval_counts.total_claims, 0), 2) AS approval_rate_pct
FROM denial_counts
INNER JOIN approval_counts ON denial_counts.insurance_provider = approval_counts.insurance_provider
ORDER BY approval_rate_pct DESC;


3.5 Clinical Quality Metrics

Healthcare quality indicators including readmission rates, length of stay trends, and clinical outcomes.

In [0]:
%sql
CREATE OR REPLACE TABLE gold_hospital.clinical_quality_metrics AS
WITH encounter_metrics AS (
  SELECT
    e.department,
    d.diagnosis_description,
    d.chronic_flag,
    COUNT(DISTINCT e.encounter_id) AS total_encounters,
    COUNT(DISTINCT e.patient_id) AS unique_patients,
    AVG(e.length_of_stay) AS avg_los,
    SUM(CASE WHEN e.readmitted_flag = 'Yes' THEN 1 ELSE 0 END) AS readmissions,
    COUNT(CASE WHEN e.status = 'Discharged' THEN 1 END) AS discharged_count
  FROM silver_hospital.encounters e
  LEFT JOIN silver_hospital.diagnoses d ON e.encounter_id = d.encounter_id
  GROUP BY e.department, d.diagnosis_description, d.chronic_flag
),
lab_metrics AS (
  SELECT
    e.department,
    COUNT(DISTINCT lt.lab_id) AS total_lab_tests,
    COUNT(CASE WHEN lt.test_result = 'Abnormal' THEN 1 END) AS abnormal_results
  FROM silver_hospital.encounters e
  LEFT JOIN silver_hospital.lab_tests lt ON e.encounter_id = lt.encounter_id
  GROUP BY e.department
)
SELECT
  em.department,
  em.diagnosis_description,
  em.chronic_flag,
  em.total_encounters,
  em.unique_patients,
  ROUND(em.avg_los, 2) AS avg_length_of_stay,
  em.readmissions,
  ROUND(em.readmissions * 100.0 / NULLIF(em.total_encounters, 0), 2) AS readmission_rate_pct,
  COALESCE(lm.total_lab_tests, 0) AS total_lab_tests,
  COALESCE(lm.abnormal_results, 0) AS abnormal_lab_results,
  ROUND(COALESCE(lm.abnormal_results, 0) * 100.0 / NULLIF(lm.total_lab_tests, 0), 2) AS abnormal_lab_rate_pct
FROM encounter_metrics em
LEFT JOIN lab_metrics lm ON em.department = lm.department
WHERE em.diagnosis_description IS NOT NULL
ORDER BY em.total_encounters DESC;

SELECT * FROM gold_hospital.clinical_quality_metrics LIMIT 20;

3.6 Denial Analysis

Detailed analysis of claim denials including reasons, appeal outcomes, and financial impact.

In [0]:
%sql
CREATE OR REPLACE TABLE gold_hospital.denial_analysis AS
WITH denial_details AS (
  SELECT
    d.denial_reason_code,
    d.denial_reason_description,
    cb.insurance_provider,
    DATE_TRUNC('month', d.denial_date) AS denial_month,
    COUNT(DISTINCT d.denial_id) AS total_denials,
    SUM(d.denied_amount) AS total_denied_amount,
    COUNT(CASE WHEN d.appeal_filed = 'Yes' THEN 1 END) AS appeals_filed,
    COUNT(CASE WHEN d.final_outcome = 'Approved' THEN 1 END) AS appeals_won,
    COUNT(CASE WHEN d.final_outcome = 'Denied' THEN 1 END) AS appeals_lost,
    COUNT(CASE WHEN d.appeal_status = 'Pending' THEN 1 END) AS appeals_pending,
    AVG(DATEDIFF(d.appeal_resolution_date, d.denial_date)) AS avg_days_to_resolution
  FROM silver_hospital.denials d
  LEFT JOIN silver_hospital.claims_and_billing cb ON d.claim_id = cb.claim_id
  WHERE d.denial_date IS NOT NULL
  GROUP BY 
    d.denial_reason_code,
    d.denial_reason_description,
    cb.insurance_provider,
    DATE_TRUNC('month', d.denial_date)
)
SELECT
  denial_month,
  denial_reason_code,
  denial_reason_description,
  insurance_provider,
  total_denials,
  ROUND(total_denied_amount, 2) AS total_denied_amount,
  appeals_filed,
  ROUND(appeals_filed * 100.0 / NULLIF(total_denials, 0), 2) AS appeal_rate_pct,
  appeals_won,
  appeals_lost,
  appeals_pending,
  ROUND(appeals_won * 100.0 / NULLIF(appeals_filed, 0), 2) AS appeal_success_rate_pct,
  ROUND(avg_days_to_resolution, 1) AS avg_resolution_days
FROM denial_details
ORDER BY denial_month DESC, total_denied_amount DESC;

SELECT * FROM gold_hospital.denial_analysis LIMIT 20;

In [0]:
%sql
-- Verify all gold layer tables were created
SHOW TABLES IN gold_hospital;